# 15.8 双塔模型 / Two-Tower Retrieval

**中文**：到目前为止的 DeepFM/DCN 等都是**排序（ranking）模型**——它们把用户特征和物品特征**拼在一起**送进网络做充分交叉，精度高，但**必须对每个候选物品单独前向一次**。物品上百万时，这样根本算不过来。工业推荐因此分两阶段：**召回（retrieval）** 先从百万物品里快速选出几百个候选，**排序** 再精排。本节的**双塔模型（Two-Tower / Dual-Encoder）** 就是召回阶段的主力。
**English**: Models like DeepFM/DCN are **ranking** models — they **concatenate** user and item features into one network for full crossing: accurate, but they require **one forward pass per candidate item**. With millions of items this is intractable. Industrial recsys therefore has two stages: **retrieval** quickly narrows millions of items to a few hundred candidates, then **ranking** re-scores them precisely. This section's **Two-Tower (Dual-Encoder)** model is the workhorse of retrieval.

---

**中文**：双塔的结构：
**English**: The two-tower structure:

```
用户特征 user features → [用户塔 user tower] → u ∈ R^d  ┐
                                                          ├── 打分 score = u · v
物品特征 item features → [物品塔 item tower] → v ∈ R^d  ┘
```

**中文**：两座塔**完全独立**，只在最后用一个**点积**相遇。这看似简单，却是它能做召回的全部秘密：
**English**: The two towers are **fully independent**, meeting only at a final **dot product**. This simplicity is the entire secret behind its retrieval ability:

**中文**：
1. **物品向量可离线预计算**：百万物品的 $v$ 提前算好、建索引。
2. **线上只算一次用户向量** $u$，再做**近似最近邻（ANN）** 检索（如 HNSW/IVF，见 12.13），毫秒级从百万物品里取回 top-K。
3. **代价是"晚交互（late interaction）"**：用户和物品在点积之前**没有任何特征交叉**，表达力弱于排序模型——所以它只做粗筛召回，不做精排。

**English**:
1. **Item vectors are precomputed offline**: all $v$ for millions of items are computed and indexed ahead of time.
2. **Online you compute $u$ once**, then run **Approximate Nearest Neighbor (ANN)** search (HNSW/IVF, see 12.13) to fetch top-K from millions in milliseconds.
3. **The cost is "late interaction"**: user and item have **no feature crossing** before the dot product, so expressiveness is weaker than a ranking model — hence it does coarse retrieval, not fine ranking.

**中文**：训练用 **batch 内采样 softmax（in-batch sampled softmax）**：一个 batch 里有 $B$ 个 (用户, 正物品) 对，对每个用户，把**同 batch 里其他用户的正物品当作负样本**，做一个 $B$ 类 softmax。这样**零额外采样成本**就得到了 $B-1$ 个负样本。
**English**: Training uses **in-batch sampled softmax**: a batch has $B$ (user, positive item) pairs; for each user, the **positive items of the other users in the batch serve as negatives**, forming a $B$-way softmax. This yields $B-1$ negatives at **zero extra sampling cost**.

$$\mathcal{L} = -\frac{1}{B}\sum_{i=1}^{B}\log\frac{\exp(\mathbf u_i\!\cdot\!\mathbf v_i/\tau)}{\sum_{j=1}^{B}\exp(\mathbf u_i\!\cdot\!\mathbf v_j/\tau)}$$

**中文**：$\tau$ 是温度（控制分布尖锐度）。但这里藏着一个**致命的坑**，本节会亲手踩一遍再修好。
**English**: $\tau$ is temperature (controls sharpness). But there is a **fatal pitfall** here, which we will hit and then fix by hand.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 召回必考）**
> **中文**：**双塔 = 用户塔 ∥ 物品塔 + 点积**，专为**召回**：物品向量离线建 ANN 索引、用户向量线上算、毫秒级取 top-K。**晚交互**是它精度不如排序模型的根因。训练用 **in-batch softmax** 省负采样。**核心坑——采样偏差**：热门物品当负样本的概率高，会被**过度惩罚**，导致召回崩盘；修法是 **logQ 校正**（从 logit 里减去 $\log Q_j$，$Q_j$≈物品被采样为负的概率∝流行度），即 Google 的 "sampling-bias-corrected" 双塔。
> **English**: **Two-Tower = user tower ∥ item tower + dot product**, built for **retrieval**: item vectors indexed offline (ANN), user vector computed online, top-K in milliseconds. **Late interaction** is why it trails ranking models. Trained with **in-batch softmax** to avoid sampling. **Key pitfall — sampling bias**: popular items appear as negatives more often and get **over-penalized**, collapsing recall; the fix is **logQ correction** (subtract $\log Q_j$ from the logit, $Q_j$ ≈ item's negative-sampling prob ∝ popularity) — Google's "sampling-bias-corrected" two-tower.


In [ ]:

# ============================================================
# 数据：MovieLens 隐式正样本 + 用户/物品特征 / implicit positives + side features
# ============================================================
import os, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as Fnn, matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
R=os.path.expanduser("~/.cache/dsfs_recsys/ml-100k")
rat=pd.read_csv(os.path.join(R,"u.data"),sep="\t",names=["user","item","rating","ts"])
usr=pd.read_csv(os.path.join(R,"u.user"),sep="|",names=["user","age","gender","occ","zip"]).set_index("user")
GEN=["unknown","Action","Adventure","Animation","Children","Comedy","Crime","Documentary","Drama",
     "Fantasy","FilmNoir","Horror","Musical","Mystery","Romance","SciFi","Thriller","War","Western"]
mv=pd.read_csv(os.path.join(R,"u.item"),sep="|",encoding="latin-1",header=None,
               names=["item","title","date","v","url"]+GEN).set_index("item")
# 时间切分 / temporal split
rs=rat.sort_values("ts"); trp=[];tep=[]
for _,g in rs.groupby("user"):
    c=int(len(g)*0.8); trp.append(g.iloc[:c]); tep.append(g.iloc[c:])
train=pd.concat(trp); test=pd.concat(tep)
uids=np.sort(rat["user"].unique()); iids=np.sort(rat["item"].unique())
u2x={u:i for i,u in enumerate(uids)}; i2x={i:j for j,i in enumerate(iids)}
m,n=len(uids),len(iids)
# 用户侧特征：性别/年龄段/职业 / user side features
gmap={"M":0,"F":1}; occs=sorted(usr["occ"].unique()); omap={o:i for i,o in enumerate(occs)}
UG=torch.tensor([gmap[usr.loc[u,"gender"]] for u in uids])
UA=torch.tensor([int(np.clip(usr.loc[u,"age"]//10,0,7)) for u in uids])
UO=torch.tensor([omap[usr.loc[u,"occ"]] for u in uids])
# 物品侧特征：类型 multi-hot / item genre multi-hot
GM=torch.tensor(np.stack([mv.loc[i,GEN].values.astype(np.float32) for i in iids]))
def pos(df):
    d=df[df["rating"]>=4]; return np.array([u2x[u] for u in d["user"]]),np.array([i2x[i] for i in d["item"]])
trU,trI=pos(train); teU,teI=pos(test)
seen=[set() for _ in range(m)]
for u,i in zip(trU,trI): seen[u].add(i)
test_pos=[[] for _ in range(m)]
for u,i in zip(teU,teI):
    if i not in seen[u]: test_pos[u].append(i)
pop=np.bincount(trI,minlength=n).astype(float)
print(f"用户 {m}, 物品 {n}, 训练正样本 {len(trU)}, 测试正样本 {sum(len(t) for t in test_pos)}")


In [ ]:

# ============================================================
# 双塔模型 + batch 内 softmax 训练 + 评估 / two towers, in-batch softmax, eval
# ============================================================
class TwoTower(nn.Module):
    def __init__(s, d=32):
        super().__init__()
        # 用户塔：id + 性别 + 年龄 + 职业 → MLP / user tower
        s.ue=nn.Embedding(m,d); s.ge=nn.Embedding(2,8); s.ae=nn.Embedding(8,8); s.oe=nn.Embedding(len(occs),8)
        s.umlp=nn.Sequential(nn.Linear(d+24,64),nn.ReLU(),nn.Linear(64,d))
        # 物品塔：id + 类型 → MLP / item tower
        s.ie=nn.Embedding(n,d); s.gmlp=nn.Sequential(nn.Linear(len(GEN),16),nn.ReLU())
        s.imlp=nn.Sequential(nn.Linear(d+16,64),nn.ReLU(),nn.Linear(64,d))
    def user(s,u): return s.umlp(torch.cat([s.ue(u),s.ge(UG[u]),s.ae(UA[u]),s.oe(UO[u])],1))
    def item(s,i): return s.imlp(torch.cat([s.ie(i),s.gmlp(GM[i])],1))

def recall_at(model, Ks=(10,50)):
    model.eval()
    with torch.no_grad():
        IV=Fnn.normalize(model.item(torch.arange(n)),dim=1)        # 预计算所有物品向量 / precompute item vecs
        out={K:0.0 for K in Ks}; c=0
        for u in range(m):
            if not test_pos[u]: continue
            uv=Fnn.normalize(model.user(torch.tensor([u])),dim=1)
            sc=(IV@uv.T).squeeze(1).numpy()
            for i in seen[u]: sc[i]=-1e9                            # 屏蔽训练已见 / mask seen
            for K in Ks:
                top=set(np.argpartition(sc,-K)[-K:].tolist())
                out[K]+=len(top&set(test_pos[u]))/len(test_pos[u])
            c+=1
    return {K:out[K]/c for K in Ks}

q=np.maximum(pop/pop.sum(),1e-8); logq=torch.tensor(np.log(q),dtype=torch.float32)   # 采样概率 logQ
def train_tt(use_logq=False, bs=256, ep=20, lr=1e-3, temp=0.1):
    torch.manual_seed(0); model=TwoTower(); opt=torch.optim.Adam(model.parameters(),lr)
    U=torch.tensor(trU); I=torch.tensor(trI); Np=len(trU)
    for e in range(ep):
        perm=torch.randperm(Np)
        for b in range(0,Np,bs):
            ix=perm[b:b+bs]
            uv=Fnn.normalize(model.user(U[ix]),dim=1); iv=Fnn.normalize(model.item(I[ix]),dim=1)
            logits=uv@iv.T/temp                                    # batch 内打分 (B,B) / in-batch logits
            if use_logq: logits=logits-logq[I[ix]].unsqueeze(0)    # logQ 校正：减去列物品的 log 采样概率
            loss=Fnn.cross_entropy(logits, torch.arange(len(ix)))  # 对角线是正样本 / diagonal = positive
            opt.zero_grad(); loss.backward(); opt.step()
    return model

# 朴素 in-batch softmax（不校正）/ naive in-batch softmax (no correction)
naive=train_tt(use_logq=False); rn=recall_at(naive)
# 流行度基线 / popularity baseline
order=np.argsort(pop)[::-1]
def pop_recall(K):
    s=0.0;c=0
    for u in range(m):
        if not test_pos[u]:continue
        rec=[i for i in order if i not in seen[u]][:K]; s+=len(set(rec)&set(test_pos[u]))/len(test_pos[u]); c+=1
    return s/c
rp={K:pop_recall(K) for K in (10,50)}
print(f"{'方法/method':<28}{'Recall@10':>11}{'Recall@50':>11}")
print(f"{'Two-Tower (朴素 in-batch)':<28}{rn[10]:>11.4f}{rn[50]:>11.4f}")
print(f"{'Popularity baseline':<28}{rp[10]:>11.4f}{rp[50]:>11.4f}")


**中文**：踩坑了！**朴素的 in-batch softmax 双塔召回竟然远输给"推最热门"基线**。这不是实现 bug，而是 in-batch 采样 softmax 最著名的陷阱——**采样偏差（sampling bias）**：
**English**: We hit the pitfall! The **naive in-batch softmax two-tower badly loses to the popularity baseline**. Not an implementation bug, but the most famous trap of in-batch sampled softmax — **sampling bias**:

**中文**：batch 内的负样本来自"其他用户的正样本"，而**热门物品出现在正样本里的频率本来就高**，于是它们被当作负样本的概率也高，被 softmax **反复往下压**。结果模型学会了"压低热门物品"，而测试集里用户真正点的恰恰大量是热门物品——召回自然崩。
**English**: In-batch negatives come from "other users' positives," and **popular items appear as positives more often**, so they are sampled as negatives more often and get **repeatedly pushed down** by the softmax. The model learns to *suppress popular items*, yet users' true test clicks are largely popular items — so recall collapses.

**中文**：修法是 **logQ 校正**：从每个候选物品的 logit 里减去 $\log Q_j$，$Q_j$ 是它被采样为负的概率（正比于流行度）。这样就抵消了"热门被过度当负样本"的偏差。这正是 Google 2019 年双塔论文《Sampling-Bias-Corrected Neural Modeling》的核心。
**English**: The fix is **logQ correction**: subtract $\log Q_j$ from each candidate's logit, where $Q_j$ is its negative-sampling probability (∝ popularity). This cancels the "popular over-sampled as negative" bias — the core of Google's 2019 two-tower paper *Sampling-Bias-Corrected Neural Modeling*.


In [ ]:

# ============================================================
# 加上 logQ 校正后重训 / retrain WITH logQ correction
# ============================================================
corr=train_tt(use_logq=True); rc=recall_at(corr)
print(f"{'方法/method':<28}{'Recall@10':>11}{'Recall@50':>11}")
print(f"{'Two-Tower 朴素 / naive':<28}{rn[10]:>11.4f}{rn[50]:>11.4f}")
print(f"{'Two-Tower + logQ 校正':<28}{rc[10]:>11.4f}{rc[50]:>11.4f}")
print(f"{'Popularity baseline':<28}{rp[10]:>11.4f}{rp[50]:>11.4f}")


**中文**：logQ 校正后，双塔召回**反超热门基线**（Recall@10 与 Recall@50 都明显更高）。一个小改动逆转了胜负——这就是为什么"召回里的采样偏差校正"是面试高频考点。下面可视化三件事。
**English**: With logQ correction, the two-tower **overtakes the popularity baseline** (both Recall@10 and Recall@50 clearly higher). A tiny change flips the outcome — why "sampling-bias correction in retrieval" is a frequent interview topic. Now three visualizations.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(15,4.3))
# ① Recall@K：朴素 vs logQ vs 热门 / recall curves
Ks=[5,10,20,50,100]
rn_c=recall_at(naive,Ks); rc_c=recall_at(corr,Ks); rp_c={K:pop_recall(K) for K in Ks}
ax[0].plot(Ks,[rn_c[K] for K in Ks],"o--",label="朴素 naive",color="#8C8C8C")
ax[0].plot(Ks,[rc_c[K] for K in Ks],"o-",label="+logQ 校正",color="#4C72B0")
ax[0].plot(Ks,[rp_c[K] for K in Ks],"^-",label="Popularity",color="#C44E52")
ax[0].set_title("Recall@K：校正后反超热门"); ax[0].set_xlabel("K"); ax[0].set_ylabel("Recall@K"); ax[0].legend()

# ② 物品向量空间(PCA) 按主类型上色 / item embedding space colored by primary genre
corr.eval()
with torch.no_grad(): IV=Fnn.normalize(corr.item(torch.arange(n)),dim=1).numpy()
idf=np.log(n/(1+GM.numpy().sum(0)))
focus=["Animation","Horror","Documentary","Western","Musical"]; fc=dict(zip(focus,["#1f77b4","#d62728","#9467bd","#ff7f0e","#2ca02c"]))
Vc=IV-IV.mean(0); _,_,Vt=np.linalg.svd(Vc,full_matrices=False); emb=Vc@Vt[:2].T
for gname in focus:
    pts=[]
    for j,iid in enumerate(iids):
        if mv.loc[iid,gname]==1:
            owned=[gg for gg in focus if mv.loc[iid,gg]==1]
            if owned and max(owned,key=lambda x:idf[GEN.index(x)])==gname: pts.append(emb[j])
    if pts: pts=np.array(pts); ax[1].scatter(pts[:,0],pts[:,1],s=12,alpha=.7,c=fc[gname],label=f"{gname}({len(pts)})")
ax[1].set_title("物品塔 embedding(PCA) / item-tower space"); ax[1].legend(fontsize=7)

# ③ batch 大小 = batch 内负样本数 的影响(均带 logQ) / batch size = #in-batch negatives
bss=[32,128,512]; r_by_bs=[recall_at(train_tt(use_logq=True,bs=bs,ep=15),(50,))[50] for bs in bss]
ax[2].plot(bss,r_by_bs,"s-",color="#55A868"); ax[2].set_xscale("log",base=2)
ax[2].set_title("batch 越大→负样本越多 / more in-batch negatives"); ax[2].set_xlabel("batch size"); ax[2].set_ylabel("Recall@50")
plt.tight_layout(); plt.savefig("/tmp/rec08_viz.png",dpi=80); plt.show()
print("logQ 校正后 Recall@10=%.4f（朴素 %.4f，热门 %.4f）"%(rc[10],rn[10],rp[10]))


**中文**：三张图的解读：
**English**: Reading the three plots:

**中文**：
1. **Recall@K 曲线**：校正后的双塔在所有 K 上都压过热门基线，而朴素双塔全程垫底——采样偏差的破坏力一目了然。
2. **物品塔 embedding 空间**：和 15.3 矩阵分解一样，双塔也从行为中学出了类型语义（动画/恐怖/纪录片等出现聚集倾向），但因为物品塔还吃了类型特征，聚集更明显——这也是双塔能缓解**物品冷启动**的原因：新物品即使没人点，靠类型特征也能落到合理位置被召回。
3. **batch 大小的影响**：in-batch 负样本数 = batch_size−1，所以**更大的 batch ≈ 更多负样本**，召回通常随之提升（工业界双塔常用很大的 batch，正是为了多负样本）。本数据规模小，提升有限甚至波动，但趋势和原理要记住。

**English**:
1. **Recall@K curves**: the corrected two-tower beats popularity at every K, while the naive one stays at the bottom throughout — the damage of sampling bias is unmistakable.
2. **Item-tower embedding space**: like MF in 15.3, the two-tower learns genre semantics from behavior (Animation/Horror/Documentary cluster), and more clearly because the item tower also ingests genre features — which is why two-tower eases **item cold-start**: a new item with no clicks still lands in a sensible spot via its features and can be retrieved.
3. **Batch size effect**: #in-batch negatives = batch_size − 1, so **larger batches ≈ more negatives**, usually lifting recall (industry uses very large batches for two-tower precisely to get more negatives). On this small dataset the gain is limited/noisy, but remember the principle.

> 💼 **实战视角 / Practical angle**
> **中文**：双塔是现代召回的事实标准（YouTube、Google Play、淘宝…）。落地要点：① 物品塔产出的向量灌进 **ANN 索引**（HNSW/IVF）做毫秒级召回；② **采样偏差校正（logQ）几乎必做**；③ 温度 $\tau$、batch 大小、是否 L2 归一化都很敏感；④ 双塔吃 side feature → **天然缓解冷启动**；⑤ 局限是**晚交互**，所以它只做召回，后面接 DeepFM/DCN 精排。面试金句：*"双塔为可检索性牺牲了交互深度；in-batch softmax 必须做 logQ 校正，否则热门物品被过度惩罚。"*
> **English**: Two-tower is the de facto standard for modern retrieval (YouTube, Google Play, Taobao…). Deployment notes: ① feed item-tower vectors into an **ANN index** (HNSW/IVF) for millisecond retrieval; ② **sampling-bias (logQ) correction is near-mandatory**; ③ temperature $\tau$, batch size, and L2-normalization are all sensitive; ④ ingesting side features → **eases cold-start naturally**; ⑤ the limit is **late interaction**, so it only retrieves, with DeepFM/DCN ranking downstream. Interview line: *"Two-tower trades interaction depth for retrievability; in-batch softmax must use logQ correction, or popular items get over-penalized."*

---
### 小结 / Summary
- **中文**：双塔=独立用户塔/物品塔+点积，为召回而生（物品向量离线 ANN、用户向量线上、毫秒 top-K）。
- **English**: Two-tower = independent user/item towers + dot product, built for retrieval (offline ANN over item vectors, online user vector, millisecond top-K).
- **中文**：in-batch softmax 省负采样，但有采样偏差——必须 logQ 校正，否则召回输给热门基线。
- **English**: In-batch softmax avoids sampling but has sampling bias — logQ correction is required, else recall loses to popularity.
- **中文**：晚交互限制精度（只召回不精排）；吃 side feature 缓解冷启动；batch 越大负样本越多。
- **English**: Late interaction caps accuracy (retrieve, don't rank); side features ease cold-start; bigger batch = more negatives.
